# Evolución del stack tecnológico según experiencia y rol
### Stack Overflow Developer Survey 2024 — Visualización de Datos (MPIA, ITA) · Entrega final

**Pregunta analítica.** ¿Cómo evoluciona el stack tecnológico y la demanda de herramientas a medida que un perfil avanza desde sus primeros años de experiencia (Entry-level, 0-2 años) hasta alcanzar roles de liderazgo o dirección (Engineering manager y Senior Executive)? La demanda se aproxima con el uso declarado en la encuesta.

**Fuente.** Stack Overflow (2024), redistribuido en Kaggle por berkayalan (2024): <https://www.kaggle.com/datasets/berkayalan/stack-overflow-annual-developer-survey-2024>. Se usa la **versión 1** fijada en `src/utils.py` (`KAGGLE_HANDLE`).

Este notebook prepara los datos del dashboard final (`docs/index.html`). No genera figuras: la visualización vive en el dashboard.

| Sección | Contenido |
|---|---|
| 1. Configuración | Librerías y parámetros |
| 2. Carga de datos | Descarga reproducible y selección de columnas |
| 3. Limpieza y variables derivadas | Experiencia, segmento de rol y rango |
| 4. Perfil de los segmentos | Tamaño y experiencia de cada segmento |
| 5. Brecha de adopción por rango de experiencia | Cálculo y exportación de los datos del dashboard |
| 6. Cifras para la narrativa | Indicadores del dashboard y valores clave |
| 7. Tamaños de muestra | n de cada pregunta, global y estratificado |

**Cómo ejecutarlo.** Ver `README.md`. El notebook corre de arriba abajo sin intervención (*Kernel → Restart & Run All*).

## 1. Configuración

In [1]:
import sys
from pathlib import Path

# Permite importar src/ al ejecutar el notebook desde notebooks/
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import pandas as pd

from src import utils
from src.utils import SEG_DEV, SEG_LIDER, SEG_OTRO, RANGOS_EXP

SEGMENTOS = [SEG_DEV, SEG_LIDER]
MIN_MUESTRA = 30  # n mínimo de cada segmento para reportar un estrato
DIR_WEB_DATOS = utils.ROOT / "docs" / "data"  # GitHub Pages publica la carpeta docs/

print(f"pandas {pd.__version__}")

pandas 3.0.2


## 2. Carga de datos

`utils.cargar_encuesta` descarga la versión fijada del dataset con `kagglehub` (queda en caché local tras la primera ejecución). Para trabajar sin conexión, define la variable de entorno `SO_SURVEY_CSV` con la ruta a una copia local de `survey_results_public.csv`.

Solo se cargan las columnas que usa el análisis. `ResponseId` es el identificador único del encuestado y sirve como llave para las tablas en formato largo.

In [2]:
PREGUNTAS = {
    "plataformas": "PlatformHaveWorkedWith",
    "herramientas": "ToolsTechHaveWorkedWith",
    "lenguajes": "LanguageHaveWorkedWith",
}
COLUMNAS = ["ResponseId", "YearsCodePro", "DevType", *PREGUNTAS.values()]
df_raw = utils.cargar_encuesta(COLUMNAS)

print(f"Registros cargados: {len(df_raw):,}")
df_raw.isna().mean().mul(100).round(1).rename("% nulos").to_frame()

Registros cargados: 65,437


,% nulos
ResponseId,0.0
YearsCodePro,21.1
DevType,9.2
LanguageHaveWorkedWith,8.7
PlatformHaveWorkedWith,35.3
ToolsTechHaveWorkedWith,19.8


## 3. Limpieza y variables derivadas

| Variable | Regla | Implementación |
|---|---|---|
| `YearsCodePro` | Texto → número. `"Less than 1 year"` → 0; `"More than 50 years"` → 51 (cota inferior) | `utils.limpiar_experiencia` |
| `Segmento` | Rol principal (primer valor de `DevType`): *Liderazgo y dirección* = Engineering manager o Senior Executive; *Desarrollo* = roles que empiezan con `Developer, ` (con coma, para excluir Developer Advocate y Developer Experience); el resto queda fuera | `utils.segmentar_rol` |
| `Rango` | Ordinal: 0-2, 3-5, 6-10, 11-15, 16-20 y 20+ años | `utils.rango_experiencia` |

No se eliminan filas aquí: cada cálculo descarta después los nulos que le afectan.

In [3]:
base = df_raw.copy()
base["YearsCodePro"] = utils.limpiar_experiencia(base["YearsCodePro"])
base["Segmento"] = utils.segmentar_rol(base["DevType"])
base["Rango"] = utils.rango_experiencia(base["YearsCodePro"])

print(f"YearsCodePro válidos: {base['YearsCodePro'].notna().sum():,} ({base['YearsCodePro'].notna().mean():.1%})")
pd.crosstab(base["Rango"], base["Segmento"], margins=True, margins_name="Total")

YearsCodePro válidos: 51,610 (78.9%)


Segmento,Desarrollo (contribuidor individual),Liderazgo y dirección,Otro,Total
Rango,,,,
0-2,6422,16,3225,9663
3-5,8232,104,2498,10834
6-10,9504,415,2734,12653
11-15,5158,424,1351,6933
16-20,3387,342,963,4692
20+,4722,497,1616,6835
Total,37425,1798,12387,51610


## 4. Perfil de los segmentos

El grupo de liderazgo tiene el doble de experiencia que el de desarrollo. Por eso la comparación directa entre segmentos mezcla el efecto del rol con el de la experiencia, y la sección 5 compara dentro de cada rango.

In [4]:
df_seg = base[base["Segmento"].isin(SEGMENTOS)]
perfil = df_seg.groupby("Segmento").agg(
    n=("ResponseId", "size"),
    con_experiencia=("YearsCodePro", "count"),
    mediana_experiencia=("YearsCodePro", "median"),
).reindex(SEGMENTOS)
perfil

,n,con_experiencia,mediana_experiencia
Segmento,,,
Desarrollo (contribuidor individual),39448,37425,7.0
Liderazgo y dirección,2112,1798,15.0


## 5. Brecha de adopción por rango de experiencia

**Método.**

- *Adopción:* proporción del grupo que usó la tecnología en el último año. El denominador es quien respondió esa pregunta: quien la dejó en blanco no cuenta como «no usa» (`utils.tabla_adopcion`).
- *Brecha:* liderazgo − desarrollo en puntos porcentuales, con IC 95% de Newcombe (híbrido de Wilson), dentro de cada rango de experiencia (`utils.diferencia_adopcion`). Se descartan los estratos con menos de 30 encuestados en cualquiera de los dos grupos, lo que elimina 0-2 años (16 líderes).
- *Global:* la misma brecha sin estratificar. El dashboard la superpone para mostrar cuándo la comparación directa engaña.
- Cada segmento lleva además su IC 95% de Wilson, para el panel de detalle.

**Selección.** Las 12 plataformas y herramientas más usadas entre los dos segmentos (contadas juntas) y los 10 lenguajes más usados entre quienes tienen experiencia válida.

**Salida.** `docs/data/brechas.json` (lo lee `docs/dashboard.js`) y `data/processed/brechas_por_rango.csv`, con el mismo contenido.

In [5]:
TOP_TECNOLOGIAS, TOP_LENGUAJES = 12, 10

# Plataformas y herramientas compiten por los mismos 12 lugares
usos_tec = pd.concat([utils.explotar_multivalor(df_seg, PREGUNTAS[p]).assign(pregunta=p)
                      for p in ["plataformas", "herramientas"]])
top_tec = usos_tec.groupby(["pregunta", "valor"]).size().nlargest(TOP_TECNOLOGIAS).reset_index()
SELECCION = {p: top_tec.loc[top_tec["pregunta"] == p, "valor"].tolist() for p in ["plataformas", "herramientas"]}
SELECCION["lenguajes"] = utils.top_valores(df_seg.dropna(subset=["Rango"]), PREGUNTAS["lenguajes"], n=TOP_LENGUAJES)


def brecha(datos: pd.DataFrame, columna: str, top: list[str], por: list[str]) -> pd.DataFrame:
    """Brecha liderazgo − desarrollo con IC de Newcombe y el IC de Wilson de cada segmento (escala 0-1)."""
    tabla = utils.tabla_adopcion(datos, columna, por=["Segmento", *por], top=top)
    dif = utils.diferencia_adopcion(tabla, por=por, min_n=MIN_MUESTRA)
    llaves = [*por, "tecnologia"]
    for seg, sufijo in [(SEG_DEV, "desarrollo"), (SEG_LIDER, "liderazgo")]:
        ic = tabla.loc[tabla["Segmento"] == seg, [*llaves, "li", "ls"]]
        dif = dif.merge(ic.rename(columns={"li": f"li_{sufijo}", "ls": f"ls_{sufijo}"}), on=llaves)
    return dif.rename(columns={"n_a": "n_desarrollo", "n_b": "n_liderazgo", "p_a": "p_desarrollo",
                               "p_b": "p_liderazgo", "d": "diferencia_pp", "li": "li_pp", "ls": "ls_pp"})


con_rango = df_seg.dropna(subset=["Rango"])
partes = []
for pregunta, columna in PREGUNTAS.items():
    top = SELECCION[pregunta]
    partes.append(brecha(df_seg, columna, top, por=[]).assign(rango="Global", pregunta=pregunta))
    partes.append(brecha(con_rango, columna, top, por=["Rango"])
                  .assign(rango=lambda x: x["Rango"].astype(str), pregunta=pregunta).drop(columns="Rango"))

COLUMNAS_PP = ["p_desarrollo", "li_desarrollo", "ls_desarrollo", "p_liderazgo", "li_liderazgo", "ls_liderazgo",
               "diferencia_pp", "li_pp", "ls_pp"]
brechas = pd.concat(partes, ignore_index=True)
# 4 decimales: la web redondea a 1 al mostrar, y redondear dos veces descuadra las restas (52.847 → 52.85 → 52.9)
brechas[COLUMNAS_PP] = (brechas[COLUMNAS_PP] * 100).round(4)
brechas["ic_excluye_cero"] = (brechas["li_pp"] > 0) | (brechas["ls_pp"] < 0)
brechas["rango"] = pd.Categorical(brechas["rango"], categories=["Global", *RANGOS_EXP], ordered=True)
brechas = brechas.sort_values(["pregunta", "tecnologia", "rango"])[
    ["tecnologia", "pregunta", "rango", "n_desarrollo", "n_liderazgo", *COLUMNAS_PP, "ic_excluye_cero"]
].reset_index(drop=True)
brechas["rango"] = brechas["rango"].astype(str)

DIR_WEB_DATOS.mkdir(parents=True, exist_ok=True)
utils.DIR_PROCESADOS.mkdir(parents=True, exist_ok=True)
brechas.to_json(DIR_WEB_DATOS / "brechas.json", orient="records", force_ascii=False, indent=0)
brechas.to_csv(utils.DIR_PROCESADOS / "brechas_por_rango.csv", index=False)

rangos_omitidos = [r for r in RANGOS_EXP if r not in set(brechas["rango"])]
print(f"Exportado: docs/data/brechas.json ({len(brechas)} filas) | rangos omitidos (n < {MIN_MUESTRA}): {rangos_omitidos}")
print({p: len(v) for p, v in SELECCION.items()})

Exportado: docs/data/brechas.json (132 filas) | rangos omitidos (n < 30): ['0-2']
{'plataformas': 3, 'herramientas': 9, 'lenguajes': 10}


**Validación.** Cada tecnología debe tener su fila global y un estrato por rango incluido, y los n de cada estrato deben ser iguales para todas las tecnologías de una misma pregunta (el denominador es por pregunta, no por tecnología).

In [6]:
rangos_incluidos = [r for r in RANGOS_EXP if r not in rangos_omitidos]
filas_por_tec = brechas.groupby(["pregunta", "tecnologia"]).size()
assert (filas_por_tec == 1 + len(rangos_incluidos)).all(), filas_por_tec[filas_por_tec != 1 + len(rangos_incluidos)]
n_distintos = brechas.groupby(["pregunta", "rango"])[["n_desarrollo", "n_liderazgo"]].nunique()
assert (n_distintos == 1).all().all(), "Los n deben depender solo de la pregunta y el rango"
assert set(SELECCION["plataformas"]) | set(SELECCION["herramientas"]) == set(top_tec["valor"])
print("Validación correcta")

Validación correcta


## 6. Cifras para la narrativa

El dashboard calcula sus indicadores en el navegador a partir de `brechas.json`. Esta celda los reproduce para poder citarlos. *Brecha consistente* = diferencia positiva con IC que excluye 0 en todos los rangos de 6 años en adelante; 3-5 se excluye del criterio porque tiene unos 90 líderes.

In [7]:
por_rango = brechas[brechas["rango"] != "Global"]
es_tec = por_rango["pregunta"] != "lenguajes"
mayor = por_rango[es_tec].loc[por_rango.loc[es_tec, "diferencia_pp"].idxmax()]
print(f"Mayor brecha (plataformas y herramientas): {mayor['tecnologia']} en {mayor['rango']} años, "
      f"{mayor['diferencia_pp']:+.1f} pp ({mayor['p_liderazgo']:.1f}% vs {mayor['p_desarrollo']:.1f}%)")

desde_6 = por_rango[por_rango["rango"] != "3-5"]
consistentes = (desde_6.assign(ok=desde_6["ic_excluye_cero"] & (desde_6["diferencia_pp"] > 0))
                .groupby(["pregunta", "tecnologia"])["ok"].all())
for grupo, preguntas in [("plataformas y herramientas", ["plataformas", "herramientas"]), ("lenguajes", ["lenguajes"])]:
    sub = consistentes[consistentes.index.get_level_values("pregunta").isin(preguntas)]
    print(f"Brecha consistente en {grupo}: {sub.sum()} de {len(sub)} → {sorted(sub[sub].index.get_level_values(1))}")

(brechas[brechas["tecnologia"].isin(["Amazon Web Services (AWS)", "Microsoft Azure", "Python", "Bash/Shell (all shells)", "SQL"])]
 .set_index(["tecnologia", "rango"])[["p_desarrollo", "p_liderazgo", "diferencia_pp", "li_pp", "ls_pp"]].round(1))

Mayor brecha (plataformas y herramientas): Amazon Web Services (AWS) en 16-20 años, +19.4 pp (76.8% vs 57.4%)
Brecha consistente en plataformas y herramientas: 5 de 12 → ['Docker', 'Google Cloud', 'Homebrew', 'Kubernetes', 'Pip']
Brecha consistente en lenguajes: 2 de 10 → ['Bash/Shell (all shells)', 'Python']


p_desarrollo  p_liderazgo  diferencia_pp  \
tecnologia                rango                                              
Bash/Shell (all shells)   Global          31.2         41.0            9.8   
                          3-5             29.8         37.9            8.1   
                          6-10            31.0         40.0            9.0   
                          11-15           33.3         45.5           12.2   
                          16-20           36.3         43.1            6.8   
                          20+             35.5         42.9            7.4   
Python                    Global          42.9         51.8            8.9   
                          3-5             47.2         62.1           15.0   
                          6-10            39.4         57.3           17.9   
                          11-15           38.0         47.7            9.7   
                          16-20           36.1         48.7           12.6   
                          20+             35.8         51.3           15.5   
SQL                       Global          53.7         58.3            4.7   
                          3-5             52.1         62.1           10.0   
                          6-10            52.3         58.3            6.0   
                          11-15           53.4         58.9            5.5   
                          16-20           59.5         57.5           -2.0   
                          20+             62.8         64.1            1.2   
Amazon Web Services (AWS) Global          53.5         68.0           14.5   
                          3-5             52.8         57.6            4.8   
                          6-10            57.5         62.7            5.2   
                          11-15           59.8         72.9           13.1   
                          16-20           57.4         76.8           19.4   
                          20+             54.3         69.0           14.8   
Microsoft Azure           Global          30.4         37.0            6.7   
                          3-5             28.1         29.3            1.3   
                          6-10            29.1         30.8            1.7   
                          11-15           32.8         32.7           -0.1   
                          16-20           36.9         38.1            1.2   
                          20+             43.3         46.7            3.3   

                                  li_pp  ls_pp  
tecnologia                rango                 
Bash/Shell (all shells)   Global    7.6   12.0  
                          3-5      -0.8   17.8  
                          6-10      4.2   13.9  
                          11-15     7.3   17.2  
                          16-20     1.3   12.3  
                          20+       2.8   12.0  
Python                    Global    6.7   11.1  
                          3-5       5.2   23.8  
                          6-10     12.9   22.7  
                          11-15     4.7   14.7  
                          16-20     7.0   18.1  
                          20+      10.9   20.1  
SQL                       Global    2.4    6.8  
                          3-5       0.3   18.9  
                          6-10      1.0   10.8  
                          11-15     0.4   10.3  
                          16-20    -7.6    3.4  
                          20+      -3.3    5.6  
Amazon Web Services (AWS) Global   12.2   16.7  
                          3-5      -5.5   14.4  
                          6-10     -0.1   10.2  
                          11-15     8.2   17.7  
                          16-20    14.0   24.2  
                          20+       9.9   19.3  
Microsoft Azure           Global    4.4    9.0  
                          3-5      -7.1   11.3  
                          6-10     -3.0    6.8  
                          11-15    -4.9    5.0  
                          16-20    -4.4    7.0  
                  

## 7. Tamaños de muestra

Encuestados que respondieron cada pregunta, sin estratificar (fila `Global`) y sumando los rangos incluidos. Las cifras se citan en el dashboard.

In [8]:
n_global = brechas[brechas["rango"] == "Global"].drop_duplicates("pregunta").set_index("pregunta")
n_estrat = por_rango.drop_duplicates(["pregunta", "rango"]).groupby("pregunta")[["n_desarrollo", "n_liderazgo"]].sum()
tamanos = n_global[["n_desarrollo", "n_liderazgo"]].join(n_estrat, rsuffix="_estratificado")
tamanos.to_csv(utils.DIR_PROCESADOS / "tamanos_muestra.csv")
tamanos

,n_desarrollo,n_liderazgo,n_desarrollo_estratificado,n_liderazgo_estratificado
pregunta,,,,
herramientas,35069,1892,27853,1620
lenguajes,38225,2054,30167,1745
plataformas,28402,1817,22990,1546
